# Fase 1 — Comprensión del problema y EDA inicial

**Proyecto:** Análisis de tarifas de Uber (Equipo 4 — Transporte)

Objetivos de este notebook:
1. Cargar el dataset original y conocer su estructura.
2. Evaluar la calidad de los datos (nulos, duplicados, valores fuera de rango).
3. Hacer una primera exploración de las variables que responde a las preguntas clave
   definidas en `docs/01_comprension_problema.md`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import FIGURES_DIR, LOCAL_TZ, NYC_BOUNDS
from src.data_loader import anomaly_summary, load_raw_data, quality_report

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.float_format", "{:,.4f}".format)

## 1. Carga del dataset

In [ ]:
df = load_raw_data()
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

In [ ]:
df.info()

## 2. Calidad de los datos

In [ ]:
quality_report(df)

In [ ]:
df.describe().T

### Registros con valores fuera de rango lógico

Se consideran anómalos: tarifas no positivas o extremas, viajes sin pasajeros o con más de 6,
coordenadas fuera del área metropolitana de Nueva York y viajes con origen igual a destino.

In [ ]:
anomaly_summary(df, NYC_BOUNDS)

## 3. Exploración inicial

Para la exploración visual se usa un filtro básico (tarifa entre 0 y 100 USD) que evita que los valores
extremos oculten la forma de las distribuciones. Las horas se convierten de UTC a la hora local de
Nueva York. La limpieza formal se realiza en la Fase 3.

In [ ]:
eda = df[(df["fare_amount"] > 0) & (df["fare_amount"] <= 100)].copy()
fecha_local = eda["pickup_datetime"].dt.tz_convert(LOCAL_TZ)  # el dataset viene en UTC
eda["anio"] = fecha_local.dt.year
eda["hora"] = fecha_local.dt.hour
eda["dia_semana"] = fecha_local.dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(eda["fare_amount"], bins=80, ax=axes[0])
axes[0].set(title="Distribución de la tarifa", xlabel="Tarifa (USD)", ylabel="Viajes")
sns.boxplot(x=eda["fare_amount"], ax=axes[1])
axes[1].set(title="Boxplot de la tarifa", xlabel="Tarifa (USD)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_distribucion_tarifa.png", dpi=120)
plt.show()

La tarifa presenta una **asimetría positiva marcada**: la mayoría de los viajes cuestan menos de 20 USD,
con una cola larga de viajes costosos. Esto anticipa que las pruebas paramétricas basadas en normalidad
podrían no ser adecuadas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
eda["anio"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set(title="Viajes por año", xlabel="Año", ylabel="Viajes")
eda["hora"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#55A868")
axes[1].set(title="Viajes por hora del día", xlabel="Hora", ylabel="Viajes")
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
eda["dia_semana"].value_counts().sort_index().set_axis(dias).plot(
    kind="bar", ax=axes[2], color="#C44E52"
)
axes[2].set(title="Viajes por día de la semana", xlabel="Día", ylabel="Viajes")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_demanda_temporal.png", dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
eda.groupby("anio")["fare_amount"].mean().plot(marker="o", ax=axes[0])
axes[0].set(title="Tarifa promedio por año", xlabel="Año", ylabel="Tarifa promedio (USD)")
eda.groupby("hora")["fare_amount"].mean().plot(marker="o", ax=axes[1], color="#55A868")
axes[1].set(title="Tarifa promedio por hora", xlabel="Hora", ylabel="Tarifa promedio (USD)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_tarifa_temporal.png", dpi=120)
plt.show()

In [ ]:
eda["passenger_count"].value_counts().sort_index().to_frame("viajes")

## 4. Conclusiones del EDA inicial

- El dataset tiene ~200.000 viajes y una cantidad muy baja de nulos, pero **sí contiene valores inválidos**
  (tarifas no positivas, coordenadas en cero o fuera de la ciudad, pasajeros en 0 o atípicos) que deben
  tratarse en la Fase 3.
- No existe una variable de distancia: se calculará a partir de las coordenadas.
- La tarifa es asimétrica a la derecha, por lo que se privilegiarán **pruebas no paramétricas**.
- Se observan patrones temporales en la demanda (horas y días) y en la tarifa promedio por año,
  lo que da pie a las hipótesis de la Fase 2.